# Pandas Part 2 — Phase 1 Cheatsheet
## Missing Values and Duplicates

### 1. Detect Missing Values
```python
df.isna()                   # Detect missing values
df.isnull()                 # Same as isna()
df.isna().sum()             # Missing count per column
df.isna().sum().sum()       # Total missing cells
df.isna().mean() * 100      # Missing percentage per column
(df.isna().mean() * 100).round(2)  # Round percentages
```

### 2. Remove Missing Values — dropna()
```python
df.dropna()                         # Drop rows with any missing value
df.dropna(axis=1)                   # Drop columns with missing values
df.dropna(how="all")                # Drop rows where all values are missing
df.dropna(subset=["Age"])           # Drop rows with missing Age
df.dropna(subset=["Age", "Salary"]) # Drop rows missing either column
df.dropna(thresh=3)                 # Keep rows with at least 3 non-missing values
df.dropna(inplace=True)             # Modify original DataFrame
```

### 3. Fill Missing Values — fillna()
```python
df.fillna(0)                              # Fill all missing values with 0
df.fillna({"Age": 25, "City": "Unknown"})  # Different values per column
df["Age"].fillna(df["Age"].mean())        # Fill with mean
df["Age"].fillna(df["Age"].median())      # Fill with median
df["City"].fillna(df["City"].mode()[0])   # Fill with most frequent value
df["City"].ffill()                        # Forward fill
df["City"].bfill()                        # Backward fill

df_filled = df.fillna(0)                  # Preserve original DataFrame
```

**Choosing a method:**
- Mean: numerical data without strong outliers.
- Median: skewed numerical data or data with outliers.
- Mode: categorical values.
- Forward/backward fill: ordered or time-series data.
- Drop rows: when removal is justified.
- Zero: only when zero is a meaningful replacement.

### 4. Detect Duplicates — duplicated()
```python
df.duplicated()                              # Check duplicate rows
df.duplicated().sum()                        # Count duplicate rows
df[df.duplicated()]                          # Display duplicate rows
df.duplicated(keep="first")                  # Default: mark later copies
df.duplicated(keep="last")                   # Mark earlier copies
df.duplicated(keep=False)                    # Mark all duplicate occurrences
df.duplicated(subset=["Name"])               # Check duplicates by Name
df.duplicated(subset=["Name", "Age"]).sum()   # Count matching combinations
```

### 5. Remove Duplicates — drop_duplicates()
```python
df.drop_duplicates()                         # Remove duplicate rows
df.drop_duplicates(keep="first")             # Keep first occurrence
df.drop_duplicates(keep="last")              # Keep last occurrence
df.drop_duplicates(keep=False)               # Remove all duplicated occurrences
df.drop_duplicates(subset=["Email"])         # Check duplicates by Email
df.drop_duplicates(subset=["Name", "Age"])   # Check selected columns
df.drop_duplicates(inplace=True)             # Modify original DataFrame

df_clean = df.drop_duplicates()              # Preserve original DataFrame
```

### 6. Important Parameters
- `axis=0`: rows; `axis=1`: columns.
- `how="any"`: drop if any value is missing.
- `how="all"`: drop only if all values are missing.
- `subset`: specify columns to check.
- `thresh`: minimum number of non-missing values required.
- `keep="first"`: keep first occurrence.
- `keep="last"`: keep last occurrence.
- `keep=False`: mark/remove every occurrence in duplicate groups.
- `inplace=True`: modify the existing DataFrame; method usually returns `None`.

### 7. Reset Index
```python
df.reset_index(drop=True)               # Return DataFrame with fresh index
df = df.reset_index(drop=True)          # Save reset DataFrame
df.reset_index(drop=True, inplace=True) # Modify existing DataFrame
```

### 8. Common Interview Mistakes
1. `df.isna().sum()` counts missing values per column; `.sum().sum()` counts all missing cells.
2. `df.isna().mean() * 100` calculates missing percentages.
3. `dropna(subset=["Age", "Salary"])` removes rows missing either selected value.
4. `duplicated(subset=["Name", "Salary"]).sum()` counts duplicates based on those columns.
5. `drop_duplicates()` and `reset_index()` do not modify the original DataFrame by default.
6. Missing values should not automatically be replaced with zero; this can distort analysis.
7. `duplicated()` returns a Boolean Series; `drop_duplicates()` returns a DataFrame by default.


In [173]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["Ayaan", "Sara", "Rahul", "Zoya", "Kabir"],
    "Age": [21, np.nan, 23, 22, np.nan],
    "Marks": [85, 90, np.nan, 76, 88],
    "City": ["Srinagar", "Delhi", None, "Mumbai", "Delhi"]
})

df

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
1,Sara,NaN,90.0,Delhi
2,Rahul,23.0,NaN,None
3,Zoya,22.0,76.0,Mumbai
4,Kabir,NaN,88.0,Delhi


>What are missing values?
Missing values occur when a dataset doesn't contain a value for a particular cell.
For example, a student dataset might have missing ages or marks.
- NaN represents a missing numeric value.
- None represents a missing value in the City column.
- Pandas recognizes both as missing values in this DataFrame.

In [174]:
# isna() and isnull()
# Both methods detect missing values. They are equivalent in Pandas.

df.isna()

,Name,Age,Marks,City
0,False,False,False,False
1,False,True,False,False
2,False,False,True,True
3,False,False,False,False
4,False,True,False,False


In [175]:
df.isnull() #same as isna()

,Name,Age,Marks,City
0,False,False,False,False
1,False,True,False,False
2,False,False,True,True
3,False,False,False,False
4,False,True,False,False


In [176]:
# Count missing values in each column
df.isna().sum()

Name     0
Age      2
Marks    1
City     1
dtype: int64

In [177]:
df.isna().sum().sum()

np.int64(4)

In [178]:
# Calculate the percentage of missing values
# This is useful when deciding whether a column has too much missing data to be useful.
# Missing % = Missing values/Total rows * 100
missing_percentage = df.isna().sum()/df.shape[0] * 100
missing_percentage = df.isna().mean()*100
missing_percentage


Name      0.0
Age      40.0
Marks    20.0
City     20.0
dtype: float64

In [179]:
# You can also display counts and percentages together:
missing_summary = pd.DataFrame({
    "Missing count":df.isna().sum(),
    "Missing percentage": df.isna().mean()*100
})
missing_summary

,Missing count,Missing percentage
Name,0,0.0
Age,2,40.0
Marks,1,20.0
City,1,20.0


> dropna()

In [180]:
# Remove missing values with dropna()
df.dropna() #removes rows with atleast one missing value

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
3,Zoya,22.0,76.0,Mumbai


In [181]:
df.dropna(axis=1) #removes column with a missing value

,Name
0,Ayaan
1,Sara
2,Rahul
3,Zoya
4,Kabir


In [182]:
df.dropna(how="all") #Drop rows only when all values are missing

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
1,Sara,NaN,90.0,Delhi
2,Rahul,23.0,NaN,None
3,Zoya,22.0,76.0,Mumbai
4,Kabir,NaN,88.0,Delhi


In [183]:
# Drop rows based on specific columns
df.dropna(subset=["Age"])
#This removes only the rows where Age is missing. Missing values in Marks or City don't matter for this operation.

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
2,Rahul,23.0,NaN,None
3,Zoya,22.0,76.0,Mumbai


In [184]:
#we can do this with multiple columns
df.dropna(subset=["Age", "City"])

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
3,Zoya,22.0,76.0,Mumbai


In [185]:
# Require a minimum number of non-missing values
df.dropna(thresh=2) #keeps rows with atleast 2 non missing values

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
1,Sara,NaN,90.0,Delhi
2,Rahul,23.0,NaN,None
3,Zoya,22.0,76.0,Mumbai
4,Kabir,NaN,88.0,Delhi


Understand inplace
df.dropna(inplace=True)
This modifies df directly rather than returning a cleaned DataFrame.
Be careful: This permanently changes the current DataFrame in your notebook session.

In [186]:
#if we want to preserve our old dataframe we ll do this
new_df = df.dropna()

> fillna()

In [187]:
# df["Age"] = df["Age"].fillna(0)
df

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
1,Sara,NaN,90.0,Delhi
2,Rahul,23.0,NaN,None
3,Zoya,22.0,76.0,Mumbai
4,Kabir,NaN,88.0,Delhi


In [188]:
df.fillna({
    "Age": 25,
    "Marks": 0,
    "City": "Unknown"
})

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
1,Sara,25.0,90.0,Delhi
2,Rahul,23.0,0.0,Unknown
3,Zoya,22.0,76.0,Mumbai
4,Kabir,25.0,88.0,Delhi


In [189]:
# Fill numeric values with the mean
# The mean is the arithmetic average.
# df["Age"] = df["Age"].fillna(df["Age"].mean())
df

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
1,Sara,NaN,90.0,Delhi
2,Rahul,23.0,NaN,None
3,Zoya,22.0,76.0,Mumbai
4,Kabir,NaN,88.0,Delhi


In [190]:
# Fill numeric values with the median
# The median is the middle value after sorting the data.
# df["Age"] = df["Age"].fillna(df["Age"].median())
df
#NOTE: Use the median when: the data is skewed or contains outliers. The median is less affected by extreme values than the mean.

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
1,Sara,NaN,90.0,Delhi
2,Rahul,23.0,NaN,None
3,Zoya,22.0,76.0,Mumbai
4,Kabir,NaN,88.0,Delhi


In [191]:
# Forward fill: ffill() carries the previous non-missing value forward.
df["Age"] = df["Age"].ffill()
df

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
1,Sara,21.0,90.0,Delhi
2,Rahul,23.0,NaN,None
3,Zoya,22.0,76.0,Mumbai
4,Kabir,22.0,88.0,Delhi


In [192]:
# Backward fill: bfill() carries the next non-missing value forward.
df["Marks"] = df["Marks"].bfill()
df

,Name,Age,Marks,City
0,Ayaan,21.0,85.0,Srinagar
1,Sara,21.0,90.0,Delhi
2,Rahul,23.0,76.0,None
3,Zoya,22.0,76.0,Mumbai
4,Kabir,22.0,88.0,Delhi


----Why shouldn't we always replace missing values with zero?----
That can distort averages, totals, business reports and machine-learning models.
Choose a method based on what the missing value means:
- Mean: reasonably balanced numerical data.
- Median: skewed numerical data or data with outliers.
- Mode: often useful for categorical data.
- Forward/backward fill: appropriate ordered or time-series data.
- Drop rows: when there are few missing records and removing them is justified.
- Keep missing values: when filling them would introduce misleading assumptions.
In practice, calculate replacement statistics from the training data alone when building predictive models, to avoid data leakage.

> Detecting and Removing Duplicates
Duplicate records occur when the same record appears more than once in a dataset. They can lead to incorrect counts, inflated totals and inaccurate analysis.

In [193]:
duplicates_df = pd.DataFrame({
    "Name": ["Ayaan", "Sara", "Rahul", "Sara", "Ayaan", "Zoya"],
    "Age": [21, 22, 23, 22, 21, 24],
    "City": ["Srinagar", "Delhi", "Mumbai", "Delhi", "Srinagar", "Pune"]
})

duplicates_df

,Name,Age,City
0,Ayaan,21,Srinagar
1,Sara,22,Delhi
2,Rahul,23,Mumbai
3,Sara,22,Delhi
4,Ayaan,21,Srinagar
5,Zoya,24,Pune


In [194]:
# Detect duplicates using duplicated()
duplicates_df.duplicated() #By default, duplicated() marks repeated rows as True, keeping the first occurrence marked as False.

0    False
1    False
2    False
3     True
4     True
5    False
dtype: bool

In [195]:
#to count duplicated rows
duplicates_df.duplicated().sum()

np.int64(2)

In [196]:
duplicates_df[duplicates_df.duplicated()]

,Name,Age,City
3,Sara,22,Delhi
4,Ayaan,21,Srinagar


In [197]:
# Understand the keep parameter
duplicates_df[duplicates_df.duplicated(keep="first")] #this is default
duplicates_df[duplicates_df.duplicated(keep="last")] #this keeps the last occurences of the duplicate and gives earlier ones
duplicates_df[duplicates_df.duplicated(keep=False)] #this returns all the rows involved in the duplications

,Name,Age,City
0,Ayaan,21,Srinagar
1,Sara,22,Delhi
3,Sara,22,Delhi
4,Ayaan,21,Srinagar


In [198]:
# Remove duplicates using drop_duplicates()
clean_duplicates = duplicates_df.drop_duplicates()
clean_duplicates

,Name,Age,City
0,Ayaan,21,Srinagar
1,Sara,22,Delhi
2,Rahul,23,Mumbai
5,Zoya,24,Pune


In [199]:
# Notice that the original index labels may remain unchanged. If you want a fresh index:
clean_duplicates = duplicates_df.drop_duplicates().reset_index(drop=True)
clean_duplicates

,Name,Age,City
0,Ayaan,21,Srinagar
1,Sara,22,Delhi
2,Rahul,23,Mumbai
3,Zoya,24,Pune


In [202]:
customers = pd.DataFrame({
    "Name": ["Ayaan", "Ayan", "Sara"],
    "Email": ["ayaan@example.com", "ayaan@example.com", "sara@example.com"],
    "City": ["Srinagar", "Delhi", "Mumbai"]
})
customers

,Name,Email,City
0,Ayaan,ayaan@example.com,Srinagar
1,Ayan,ayaan@example.com,Delhi
2,Sara,sara@example.com,Mumbai


In [203]:
# subset
customers.duplicated(subset=["Email"])

0    False
1     True
2    False
dtype: bool

In [ ]:
customers.drop_duplicates(subset=["Email"]) #remove duplicate emails

,Name,Email,City
0,Ayaan,ayaan@example.com,Srinagar
2,Sara,sara@example.com,Mumbai


In [212]:
#we can also check multiple columns
duplicates_df.duplicated(subset=["Age", "Name"])

0    False
1    False
2    False
3     True
4     True
5    False
dtype: bool

In [220]:
# You can remove duplicates directly from an existing DataFrame: inplace()
duplicates_df.drop_duplicates(inplace=True)
duplicates_df

,Name,Age,City
0,Ayaan,21,Srinagar
1,Sara,22,Delhi
2,Rahul,23,Mumbai
5,Zoya,24,Pune
